Exercícios - Transformações X Ações PySpark

  Exercício 1 — Filtragem e nova coluna

  Crie um DataFrame com os seguintes dados de produtos:

  data = [
      ("Notebook", "Eletrônicos", 3500),
      ("Camiseta", "Roupas", 80),
      ("Smartphone", "Eletrônicos", 2200),
      ("Calça", "Roupas", 150),
      ("Tablet", "Eletrônicos", 1800),
  ]
  columns = ["Produto", "Categoria", "Preco"]

  1. Filtre apenas os produtos da categoria "Eletrônicos".
  2. Adicione uma coluna "Preco_Com_Desconto" com 10% de desconto (ou seja, Preco * 0.9).
  3. Exiba o resultado com .show().

  ---
  Exercício 2 — Ordenação

  Usando o mesmo DataFrame de produtos acima (sem filtro):

  1. Selecione apenas as colunas "Produto" e "Preco".
  2. Ordene pelo preço em ordem decrescente.
  3. Exiba o resultado.

  ---
  Exercício 3 — Agrupamento e média

  Usando o DataFrame de produtos:

  1. Agrupe por "Categoria".
  2. Calcule a média de preço por categoria.
  3. Arredonde o resultado para 2 casas decimais e renomeie a coluna para "Preco_Medio".
  4. Exiba ordenado pelo "Preco_Medio" crescente.

  ---
  Exercício 4 — Contagem por grupo

  Usando o DataFrame de funcionários do notebook:

  data = [
      ("Johnshon","Vendas", 3000),
      ("Anna","Marketing", 4500),
      ("Mike", "Vendas", 3500),
      ("Sara", "Marketing", 4000),
      ("João", "Vendas", 3000)
  ]

  1. Agrupe por "Setor".
  2. Conte quantos funcionários existem em cada setor (use .count()).
  3. Renomeie a coluna gerada para "Total_Funcionarios".
  4. Exiba o resultado.

  ---
  Exercício 5 — Pipeline completo (desafio)

  Crie um DataFrame com dados de vendas mensais:

  data = [
      ("João", "Janeiro", 1500),
      ("Maria", "Janeiro", 2000),
      ("João", "Fevereiro", 1800),
      ("Maria", "Fevereiro", 2200),
      ("Carlos", "Janeiro", 900),
      ("Carlos", "Fevereiro", 1100),
  ]
  columns = ["Vendedor", "Mes", "Vendas"]

  1. Filtre apenas os registros onde "Vendas" > 1000.
  2. Adicione uma coluna "Bonus" equivalente a 5% das vendas.
  3. Agrupe por "Vendedor" e some o total de "Vendas" e "Bonus".
  4. Ordene pelo total de vendas em ordem decrescente.
  5. Exiba o resultado final.


In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [0]:
spark = SparkSession.builder.appName("0002_spark_e").getOrCreate()

In [0]:
#Exercicio 1

data_1 = [
      ("Notebook", "Eletrônicos", 3500),
      ("Camiseta", "Roupas", 80),
      ("Smartphone", "Eletrônicos", 2200),
      ("Calça", "Roupas", 150),
      ("Tablet", "Eletrônicos", 1800),
  ]
columns_1 = ["Produto", "Categoria", "Preco"]

df_1 = spark.createDataFrame(data = data_1, schema = columns_1)

#Filtro categoria
df_modify = df_1.filter(F.col("Categoria") == "Eletrônicos")

#Coluna com preço desconto
df_modify = df_modify.withColumn("Preço - 10% desconto", F.col("preco") * 0.9)

#Exibir
df_modify.show()

In [0]:
#Exercicio 2

df_2 = df_1

#Selecionado colunas
df_2 = df_2.select("Produto","Preco")

#Ordenando preço
df_2 = df_2.orderBy(F.desc(F.col("Preco")))

#Exibir
df_2.show()

In [0]:
#Exercicio 3

df_3 = df_1

#Agrupando categoria
df_3 = df_3.groupBy(F.col("Categoria"))

#Media preco
df_3 = df_3.avg("Preco")

#Renomeando coluna
df_3 = df_3.withColumnRenamed("avg(Preco)", "Preço Médio")

#Round na coluna
df_3 = df_3.withColumn("Preço Médio", F.round(F.col("Preço Médio"),2))

#Preco medio crescente
df_3 = df_3.orderBy(F.col("Preço Médio").asc())

#Exibir
df_3.show()




In [0]:
#Exercicio 4

data_4 = [
    ("Johnshon","Vendas", 3000),
    ("Anna","Marketing", 4500),
    ("Mike", "Vendas", 3500),
    ("Sara", "Marketing", 4000),
    ("João", "Vendas", 3000)
]

columns_4 = ["Nome","Setor","Salário"]

df_4 = spark.createDataFrame(data=data_4, schema=columns_4)

#Agrupando por setor
df_4 = df_4.groupBy(F.col("Setor"))

#Contagem funcionário por setor
df_4 = df_4.count()

#Renomeando coluna de contagem
df_4 = df_4.withColumnRenamed("count", "Total_Funcionários")

#Exibir
df_4.show()

In [0]:
#Exercicio 5 (Pipeline)

data_5 = [
    ("João", "Janeiro", 1500),
    ("Maria", "Janeiro", 2000),
    ("João", "Fevereiro", 1800),
    ("Maria", "Fevereiro", 2200),
    ("Carlos", "Janeiro", 900),
    ("Carlos", "Fevereiro", 1100),
]

columns_5 = ["Vendedor", "Mes", "Vendas"]

df_5 = spark.createDataFrame(data=data_5, schema=columns_5)

#Filtro de vendas > 1000
df_5 = df_5.filter(F.col("Vendas") > 1000)

#Adicionar coluna "Bonus", com 5% de vendas
df_5 = df_5.withColumn("Bonus", F.col("Vendas") * 5/100)

#Agrupar por vendedor, somar vendas e bonus
df_5 = df_5.groupBy(F.col("Vendedor")).sum("Vendas", "Bonus")

#Renomear Colunas
df_5 = df_5.withColumnRenamed("sum(Vendas)", "Total Vendas")
df_5 = df_5.withColumnRenamed("sum(Bonus)","Total Bonus")

#Ordenando pelo total vendas decrescente
df_5 = df_5.orderBy(F.col("Total Vendas").desc())

#Exibir
df_5.show()